# Imputation Demo
Demonstrate imputation methods on a small dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from imputation_methods import (
    MeanImputer,
    MedianImputer,
    KNNImputer,
    PMMImputer,
    MICEImputer,
    rmse,
    mae,
)

In [ ]:
data = load_diabetes(as_frame=True).frame
# keep only a subset of columns for clarity
data = data.iloc[:, :5]
original = data.copy()
# introduce 10% missing values
mask = np.random.RandomState(0).rand(*data.shape) < 0.1
data[mask] = np.nan
data.head()

In [ ]:
# visualize missingness
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
sns.heatmap(data.isna(), cbar=False)
plt.show()

In [ ]:
mean_imp = MeanImputer().impute(data)
median_imp = MedianImputer().impute(data)
knn_imp = KNNImputer(n_neighbors=3).impute(data)
pmm_imp = PMMImputer(n_neighbors=3, random_state=0).impute(data)
mice_imp = MICEImputer(random_state=0).impute(data)

In [ ]:
# Score only the cells that were hidden; observed cells are copied unchanged.
missing = data.isna().to_numpy()
true_values = pd.Series(original.to_numpy()[missing])

def score(imputed):
    return rmse(true_values, pd.Series(imputed.to_numpy()[missing]))

results = pd.DataFrame({
    'mean': score(mean_imp),
    'median': score(median_imp),
    'knn': score(knn_imp),
    'pmm': score(pmm_imp),
    'mice': score(mice_imp),
}, index=['RMSE']).T
results

## Pros and Cons
- **Mean/Median**: Simple and fast but can distort variability.
- **KNN**: Captures local structure but requires choosing k.
- **PMM**: Preserves distribution but is computationally heavier.
- **MICE**: Flexible but may be slow on large data.